In [ ]:
!pip install -q transformers datasets tf-keras

In [ ]:
import os

# --- CRITICAL FIX FOR TENSORFLOW 2.16+ ---
# This forces TensorFlow to use the legacy Keras 2 backend, which
# is compatible with Hugging Face Transformers.
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification
from google.colab import files

In [ ]:
BATCH_SIZE = 16
MAX_LEN = 256
LEARNING_RATE = 2e-5
EPOCHS = 2

# Verify that we are running with the fix
print(f"TensorFlow Version: {tf.__version__}")
try:
    # Check if legacy keras is active
    import tf_keras
    print("Success: Using Legacy Keras (tf-keras) compatibility.")
except ImportError:
    print("Warning: tf-keras not found. Code might fail.")

TensorFlow Version: 2.19.0
Success: Using Legacy Keras (tf-keras) compatibility.


In [ ]:
print("Loading IMDB dataset...")
dataset = load_dataset("imdb")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

Loading IMDB dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

...

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN
    )

print("Tokenizing data...")
tokenized_datasets = dataset.map(tokenize_batch, batched=True)

Tokenizing data...


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format(type="tensorflow", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
train_tf = tokenized_datasets["train"].to_tf_dataset(
    columns=["input_ids", "attention_mask"],
    label_cols=["labels"],
    shuffle=True,
    batch_size=BATCH_SIZE,
)

test_tf = tokenized_datasets["test"].to_tf_dataset(
    columns=["input_ids", "attention_mask"],
    label_cols=["labels"],
    shuffle=False,
    batch_size=BATCH_SIZE,
)

/usr/local/lib/python3.12/dist-packages/datasets/arrow_dataset.py:403: FutureWarning: The output of `to_tf_dataset` will change when a passing single element list for `labels` or `columns` in the next datasets version. To return a tuple structure rather than dict, pass a single string.
Old behaviour: columns=['a'], labels=['labels'] -> (tf.Tensor, tf.Tensor)  
             : columns='a', labels='labels' -> (tf.Tensor, tf.Tensor)  
New behaviour: columns=['a'],labels=['labels'] -> ({'a': tf.Tensor}, {'labels': tf.Tensor})  
             : columns='a', labels='labels' -> (tf.Tensor, tf.Tensor) 
  warnings.warn(


In [ ]:
print("Initializing Model...")
model = TFAutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2,
    from_pt=True
)

Initializing Model...


pytorch_model.bin:   0%|          | 0.00/268M [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['vocab_layer_norm.bias', 'vocab_projector.weight', 'vocab_projector.bias', 'vocab_transform.bias', 'vocab_transform.weight', 'vocab_layer_norm.weight']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForSeq

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.metrics import SparseCategoricalAccuracy

model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss=SparseCategoricalCrossentropy(from_logits=True),
    metrics=[SparseCategoricalAccuracy(name="accuracy")]
)
model.summary()

Model: "tf_distil_bert_for_sequence_classification"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 distilbert (TFDistilBertMa  multiple                  66362880  
 inLayer)                                                        
                                                                 
 pre_classifier (Dense)      multiple                  590592    
                                                                 
 classifier (Dense)          multiple                  1538      
                                                                 
 dropout_19 (Dropout)        multiple                  0         
                                                                 
Total params: 66955010 (255.41 MB)
Trainable params: 66955010 (255.41 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [ ]:
print("Starting Training...")
history = model.fit(
    train_tf,
    validation_data=test_tf,
    epochs=EPOCHS
)

Starting Training...
Epoch 1/2
1563/1563 [==============================] - 900s 557ms/step - loss: 0.2852 - accuracy: 0.8820 - val_loss: 0.2187 - val_accuracy: 0.9117
Epoch 2/2
1563/1563 [==============================] - 868s 556ms/step - loss: 0.1666 - accuracy: 0.9380 - val_loss: 0.2188 - val_accuracy: 0.9120


In [ ]:
print("Evaluating on Test Set...")
eval_res = model.evaluate(test_tf)
print(f"Final Test Loss: {eval_res[0]}")
print(f"Final Test Accuracy: {eval_res[1]}")

Evaluating on Test Set...
1563/1563 [==============================] - 222s 142ms/step - loss: 0.2188 - accuracy: 0.9120
Final Test Loss: 0.21881583333015442
Final Test Accuracy: 0.9120000004768372


In [ ]:
# Define the folder name and the zip file name
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    save_directory = "distilbert_imdb_tf_model"
else:
    save_directory = "../models/distilbert_imdb_tf_model"

zip_filename = "distilbert_imdb_tf_model.zip"

# Save the model to the specified directory
print(f"Saving the model to: {save_directory}")
model.save_pretrained(save_directory)

# Zip the entire model folder
# This is crucial because Colab cannot download folders directly,
# and zipping large files improves reliability.
print(f"Zipping the model folder: {save_directory}")
!zip -r {zip_filename} {save_directory}

# 2. Trigger the local download
print(f"Starting download of {zip_filename}...")
files.download(zip_filename)

# Optional: You can also verify the file size before downloading
print(f"\nSize of the zip file:")
!du -h {zip_filename}

Saving the model to: distilbert_imdb_tf_model
Zipping the model folder: distilbert_imdb_tf_model
  adding: distilbert_imdb_tf_model/ (stored 0%)
  adding: distilbert_imdb_tf_model/tf_model.h5 (deflated 8%)
  adding: distilbert_imdb_tf_model/config.json (deflated 42%)
Starting download of distilbert_imdb_tf_model.zip...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Size of the zip file:
236M	distilbert_imdb_tf_model.zip
